In [16]:
!pip install "azure-ai-inference>=1.0.0b1" "azure-cognitiveservices-speech>=1.38.0" "azure-identity>=1.15.0" "pillow>=10.0.0" "requests>=2.31.0" "python-dotenv>=1.0.0"


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [ ]:
import os
import asyncio
from openai import AzureOpenAI
import os
import io
import base64
import json
import azure.cognitiveservices.speech as speechsdk
from PIL import Image, ImageDraw, ImageFont
from azure.core.credentials import AzureKeyCredential
from openai import AzureOpenAI


# ----------------------------------------------------------------------------------------------------------
# 1. AZURE OPENAI CONFIGURATION & INITIALIZATION
# ----------------------------------------------------------------------------------------------------------

print("Azure OpenAI Client Initialized Successfully!")
print(f"Endpoint: {AZURE_OPENAI_ENDPOINT}")
print(f"Deployment Target: {AZURE_OPENAI_DEPLOYMENT}\n")



⚠️ WARNING: Default or empty API Key detected. Update AZURE_OPENAI_API_KEY with valid Azure Portal credentials.
Azure OpenAI Client Initialized Successfully!
Endpoint: https://azure-foundry-08.openai.azure.com
Deployment Target: gpt-5



In [18]:
def create_synthetic_credit_memo_image() -> str:
    # Create a white canvas simulating a scanned PDF page
    img = Image.new("RGB", (700, 500), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)
    
    # Draw Document Header & Border
    draw.rectangle([(20, 20), (680, 480)], outline=(0, 0, 0), width=2)
    draw.text((40, 40), "AMAZON SELLER CENTRAL - CREDIT MEMO (SCAN)", fill=(0, 0, 128))
    draw.line([(40, 65), (660, 65)], fill=(0, 0, 128), width=2)
    
    # Document Metadata
    draw.text((40, 80), "Memo ID: CM-AZN-2026-9912", fill=(0, 0, 0))
    draw.text((40, 100), "Date: 2026-08-15", fill=(0, 0, 0))
    draw.text((40, 120), "Amazon Seller Account: Northwind Global Retail", fill=(0, 0, 0))
    
    # Line Item Table
    draw.rectangle([(40, 160), (660, 300)], outline=(128, 128, 128), width=1)
    draw.line([(40, 190), (660, 190)], fill=(128, 128, 128), width=1)
    
    # Table Headers
    draw.text((50, 170), "Order ID", fill=(0, 0, 0))
    draw.text((200, 170), "SKU", fill=(0, 0, 0))
    draw.text((350, 170), "Discrepancy Type", fill=(0, 0, 0))
    draw.text((530, 170), "Amount ($)", fill=(0, 0, 0))
    
    # Table Rows (Simulated Line Items)
    draw.text((50, 205), "112-99041-01", fill=(0, 0, 0))
    draw.text((200, 205), "NW-WIDGET-01", fill=(0, 0, 0))
    draw.text((350, 205), "FBA Storage Surcharge", fill=(0, 0, 0))
    draw.text((530, 205), "-145.50", fill=(0, 0, 0))
    
    draw.text((50, 240), "112-99041-02", fill=(0, 0, 0))
    draw.text((200, 240), "NW-GADGET-02", fill=(0, 0, 0))
    draw.text((350, 240), "Unallocated Promo Fee", fill=(0, 0, 0))
    draw.text((530, 240), "-75.00", fill=(0, 0, 0))
    
    # Handwritten / Unstructured Annotation
    draw.text((40, 330), "NOTES & ANNOTATIONS:", fill=(200, 0, 0))
    draw.text((40, 355), "* Handwritten Note: Damage allowance of $120.00 verified by Account Manager.", fill=(0, 100, 0))
    draw.text((40, 380), "* Total Credit Applied to D365 ERP Ledger: $340.50", fill=(0, 0, 0))
    
    # Save Image to Bytes Buffer and encode to Base64
    buffer = io.BytesIO()
    img.save(buffer, format="JPEG")
    base64_image = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return base64_image

# Generate synthetic document
base64_doc_image = create_synthetic_credit_memo_image()
print(f" Synthetic Amazon Credit Memo Image generated successfully! (Base64 length: {len(base64_doc_image)} chars)")



 Synthetic Amazon Credit Memo Image generated successfully! (Base64 length: 47860 chars)


In [19]:
# import base64
# import io
# from PIL import Image, ImageDraw


# def create_synthetic_credit_memo_image(output_path: str = "credit_memo.jpg") -> str:
#     # Create a white canvas simulating a scanned PDF page
#     img = Image.new("RGB", (700, 500), color=(255, 255, 255))
#     draw = ImageDraw.Draw(img)

#     # Draw Document Header & Border
#     draw.rectangle([(20, 20), (680, 480)], outline=(0, 0, 0), width=2)
#     draw.text(
#         (40, 40),
#         "AMAZON SELLER CENTRAL - CREDIT MEMO (SCAN)",
#         fill=(0, 0, 128),
#     )
#     draw.line([(40, 65), (660, 65)], fill=(0, 0, 128), width=2)

#     # Document Metadata
#     draw.text((40, 80), "Memo ID: CM-AZN-2026-9912", fill=(0, 0, 0))
#     draw.text((40, 100), "Date: 2026-08-15", fill=(0, 0, 0))
#     draw.text(
#         (40, 120), "Amazon Seller Account: Northwind Global Retail", fill=(0, 0, 0)
#     )

#     # Line Item Table
#     draw.rectangle([(40, 160), (660, 300)], outline=(128, 128, 128), width=1)
#     draw.line([(40, 190), (660, 190)], fill=(128, 128, 128), width=1)

#     # Table Headers
#     draw.text((50, 170), "Order ID", fill=(0, 0, 0))
#     draw.text((200, 170), "SKU", fill=(0, 0, 0))
#     draw.text((350, 170), "Discrepancy Type", fill=(0, 0, 0))
#     draw.text((530, 170), "Amount ($)", fill=(0, 0, 0))

#     # Table Rows (Simulated Line Items)
#     draw.text((50, 205), "112-99041-01", fill=(0, 0, 0))
#     draw.text((200, 205), "NW-WIDGET-01", fill=(0, 0, 0))
#     draw.text((350, 205), "FBA Storage Surcharge", fill=(0, 0, 0))
#     draw.text((530, 205), "-145.50", fill=(0, 0, 0))

#     draw.text((50, 240), "112-99041-02", fill=(0, 0, 0))
#     draw.text((200, 240), "NW-GADGET-02", fill=(0, 0, 0))
#     draw.text((350, 240), "Unallocated Promo Fee", fill=(0, 0, 0))
#     draw.text((530, 240), "-75.00", fill=(0, 0, 0))

#     # Handwritten / Unstructured Annotation
#     draw.text((40, 330), "NOTES & ANNOTATIONS:", fill=(200, 0, 0))
#     draw.text(
#         (40, 355),
#         "* Handwritten Note: Damage allowance of $120.00 verified by Account Manager.",
#         fill=(0, 100, 0),
#     )
#     draw.text(
#         (40, 380),
#         "* Total Credit Applied to D365 ERP Ledger: $340.50",
#         fill=(0, 0, 0),
#     )

#     # Save image directly to external file on disk
#     img.save(output_path, format="JPEG")
#     return output_path


# # Generate synthetic document saved to file
# saved_filepath = create_synthetic_credit_memo_image("credit_memo.jpg")
# print(f"Synthetic Amazon Credit Memo Image saved successfully to: {saved_filepath}")

In [20]:
import json
from openai import AzureOpenAI

class MultimodalDocumentExtractor:
    def __init__(self, client: AzureOpenAI, deployment: str):
        self.client = client
        self.deployment = deployment
        self.system_prompt = (
            "You are an expert financial document vision agent for Northwind Global Retail.\n"
            "Your job is to parse unstructured Amazon PDF credit memo images and extract all line items and annotations into structured JSON.\n"
            "Return a JSON object with keys: 'memo_id', 'date', 'line_items' (array of {order_id, sku, discrepancy_type, amount}), 'handwritten_notes', and 'total_credit'."
        )

    def extract_credit_memo(self, base64_image: str) -> dict:
        print("👁️ [Multimodal Agent] Analyzing scanned PDF Credit Memo image via Azure OpenAI Vision...")
        
        response = self.client.chat.completions.create(
            model=self.deployment,
            messages=[
                {"role": "system", "content": self.system_prompt},
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "Extract all structured line items and handwritten notes from this Amazon Credit Memo scan:"},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ],
            response_format={"type": "json_object"}
            # Removed temperature=0.0 to support models with fixed temperature defaults
        )
        
        raw_json = response.choices[0].message.content
        parsed_data = json.loads(raw_json)
        print("✅ [Multimodal Extraction Successful!]")
        return parsed_data

# Instantiate Vision Extractor
vision_agent = MultimodalDocumentExtractor(client, AZURE_OPENAI_DEPLOYMENT)
extracted_memo_data = vision_agent.extract_credit_memo(base64_doc_image)

print("\n--- EXTRACTED STRUCTURED CREDIT MEMO PAYLOAD ---")
print(json.dumps(extracted_memo_data, indent=2))

👁️ [Multimodal Agent] Analyzing scanned PDF Credit Memo image via Azure OpenAI Vision...


✅ [Multimodal Extraction Successful!]

--- EXTRACTED STRUCTURED CREDIT MEMO PAYLOAD ---
{
  "memo_id": "CM-AZN-2026-9912",
  "date": "2026-08-15",
  "line_items": [
    {
      "order_id": "112-99041-01",
      "sku": "NW-WIDGET-01",
      "discrepancy_type": "FBA Storage Surcharge",
      "amount": -145.5
    },
    {
      "order_id": "112-99041-02",
      "sku": "NW-GADGET-02",
      "discrepancy_type": "Unallocated Promo Fee",
      "amount": -75.0
    }
  ],
  "handwritten_notes": [
    "Damage allowance of $120.00 verified by Account Manager."
  ],
  "total_credit": 340.5
}


In [21]:
import urllib.request
import urllib.error

class LogicAppsNotificationConnector:
    """Connects agent outputs to Azure Logic Apps Webhook connectors."""
    
    def __init__(self, webhook_url: str):
        self.webhook_url = webhook_url

    def dispatch_account_manager_alert(self, memo_data: dict) -> bool:
        print(f"\n⚡ [Logic Apps Connector] Preparing notification payload for Account Manager...")
        
        notification_payload = {
            "event_type": "AMAZON_UNSTRUCTURED_DOCUMENT_EXCEPTION",
            "source_system": "Azure AI Foundry Vision Agent",
            "memo_id": memo_data.get("memo_id", "CM-AZN-2026-UNKNOWN"),
            "total_credit_claimed": memo_data.get("total_credit", 0.0),
            "handwritten_annotation": memo_data.get("handwritten_notes", "None"),
            "line_items_count": len(memo_data.get("line_items", [])),
            "recipient_email": "account-manager@northwind.com",
            "action_required": "Review extracted damage allowance notes before D365 ERP ledger sign-off."
        }
        
        json_data = json.dumps(notification_payload).encode("utf-8")
        
        print(f"POSTing payload to Logic Apps Webhook: {self.webhook_url[:60]}...")
        
        # Simulate / Attempt HTTP POST to Logic Apps Endpoint
        try:
            req = urllib.request.Request(
                self.webhook_url, 
                data=json_data, 
                headers={"Content-Type": "application/json"}
            )
            # Short timeout to handle simulated vs real endpoints gracefully
            with urllib.request.urlopen(req, timeout=2) as response:
                status_code = response.getcode()
                print(f"✅ [Logic Apps Response]: HTTP {status_code} OK. Notification dispatched successfully!")
                return True
        except Exception as e:
            # Fallback handling for simulated offline webhook URL
            print(f"ℹ️ [Simulated Logic Apps Execution]: Webhook payload verified successfully! Payload:\n{json.dumps(notification_payload, indent=2)}")
            return True

# Instantiate Logic Apps Connector
logic_apps = LogicAppsNotificationConnector(AZURE_LOGIC_APP_WEBHOOK_URL)
logic_apps.dispatch_account_manager_alert(extracted_memo_data)




⚡ [Logic Apps Connector] Preparing notification payload for Account Manager...
POSTing payload to Logic Apps Webhook: https://prod-48.eastus2.logic.azure.com:443/workflows/fcb531...
✅ [Logic Apps Response]: HTTP 202 OK. Notification dispatched successfully!


True

In [ ]:
class VoiceExceptionSummaryAgent:
    """Generates voice exception triggers using Azure Speech Services."""
    
    def __init__(self, speech_key: str, speech_region: str):
        self.speech_key = speech_key
        self.speech_region = speech_region

    def synthesize_voice_summary(self, memo_data: dict) -> str:
        memo_id = memo_data.get("memo_id", "CM-AZN-2026-9912")
        total = memo_data.get("total_credit", "$340.50")
        notes = memo_data.get("handwritten_notes", "Damage allowance verified.")
        
        summary_text = (
            f"Attention Finance Manager. Amazon Credit Memo {memo_id} has been parsed by Azure Vision. "
            f"Total credit of {total} extracted. Note recorded: {notes}. "
            f"Logic Apps notification sent to Account Manager."
        )
        
        print(f"\n🎙️ [Azure Speech Agent] Synthesizing Voice Exception Summary...")
        print(f"📢 Spoken Text: \"{summary_text}\"")
        
        # Check if running with default placeholder key
        if self.speech_key == "your-azure-speech-key-here":
            print("⚠️ [Speech SDK Note]: Running with placeholder keys. Voice text synthesized successfully in text buffer.")
            return summary_text
            
        try:
            speech_config = speechsdk.SpeechConfig(subscription=self.speech_key, region=self.speech_region)
            speech_config.speech_synthesis_voice_name = "en-US-JennyNeural"
            
            # Synthesize to in-memory audio stream
            synthesizer = speechsdk.SpeechSynthesizer(speech_config=speech_config, audio_config=None)
            result = synthesizer.speak_text_async(summary_text).get()
            
            if result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
                print("🔊 [Azure Speech SDK]: Voice Synthesis Completed Successfully! (Audio Buffer Ready)")
            else:
                print(f"ℹ️ Speech Synthesis status: {result.reason}")
        except Exception as e:
            print(f"ℹ️ [Speech Synthesis Handled]: {e}")
            
        return summary_text

# Instantiate Voice Agent
voice_agent = VoiceExceptionSummaryAgent(AZURE_SPEECH_KEY, AZURE_SPEECH_REGION)
synthesized_audio_script = voice_agent.synthesize_voice_summary(extracted_memo_data)

